# Forest Fire Detection Using CNN and ResNet-50

This notebook trains an image classifier for forest fire detection using transfer learning with ResNet-50. It follows the project report: binary classification, 224 x 224 images, data augmentation, Adam optimizer, and evaluation with accuracy, classification report, and confusion matrix.

Classes:

- fire
- no_fire


## 1. Setup

Run the install cell only if the required libraries are not already installed in your Jupyter environment.


In [ ]:
# Optional first-time setup
# %pip install -r ../requirements.txt

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## 2. Dataset Paths

Place the image dataset in this structure before training:

```text
data/
|-- train/
|   |-- fire/
|   `-- no_fire/
`-- test/
    |-- fire/
    `-- no_fire/
```


In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001

PROJECT_ROOT = Path("..").resolve()
TRAIN_DIR = PROJECT_ROOT / "data" / "train"
TEST_DIR = PROJECT_ROOT / "data" / "test"
MODEL_OUT = PROJECT_ROOT / "models" / "forest_fire_resnet50.keras"
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Train folder:", TRAIN_DIR)
print("Test folder:", TEST_DIR)

In [ ]:
def check_dataset(train_dir=TRAIN_DIR, test_dir=TEST_DIR):
    required = [
        train_dir / "fire",
        train_dir / "no_fire",
        test_dir / "fire",
        test_dir / "no_fire",
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Dataset folders are missing. Create these folders and add images:
" + "
".join(missing)
        )

check_dataset()

## 3. Data Loading and Augmentation

The training generator applies augmentation so the model sees varied image conditions. The validation/test generator only applies ResNet-50 preprocessing.


In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

print("Class mapping:", train_generator.class_indices)

## 4. Build ResNet-50 Transfer Learning Model

The base ResNet-50 layers are frozen first. A new binary classification head is added for fire/no-fire prediction.


In [ ]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 5. Train the Model

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint(MODEL_OUT, monitor="val_loss", save_best_only=True),
]

history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 6. Training Curves

In [ ]:
def plot_history(history):
    metrics = history.history
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(metrics["accuracy"], label="train accuracy")
    plt.plot(metrics["val_accuracy"], label="validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(metrics["loss"], label="train loss")
    plt.plot(metrics["val_loss"], label="validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_history(history)

## 7. Evaluate the Model

In [ ]:
loss, accuracy = model.evaluate(test_generator)
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")

probabilities = model.predict(test_generator).ravel()
predictions = (probabilities >= 0.5).astype(int)
labels = list(test_generator.class_indices.keys())

print(classification_report(test_generator.classes, predictions, target_names=labels))

In [ ]:
matrix = confusion_matrix(test_generator.classes, predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 8. Predict a Single Image

Change `IMAGE_PATH` to any image you want to test.


In [ ]:
def prepare_image(image_path):
    img = image.load_img(image_path, target_size=IMAGE_SIZE)
    array = image.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    return preprocess_input(array)

def predict_fire(image_path, model_path=MODEL_OUT, threshold=0.5):
    trained_model = load_model(model_path)
    prepared = prepare_image(image_path)
    probability = float(trained_model.predict(prepared)[0][0])
    label = "Fire" if probability >= threshold else "No Fire"
    return label, probability

# Example:
# IMAGE_PATH = PROJECT_ROOT / "data" / "test" / "fire" / "sample.jpg"
# label, probability = predict_fire(IMAGE_PATH)
# print(label, probability)

## Notes

The dataset images and trained model file are not bundled here because the original CNN dataset/model were not present in the local project folder. Add the dataset locally using the folder structure above, then run this notebook from top to bottom.
